# 01 · Dataset preparation and EDA

**Why:** before we fine-tune YOLO, we turn our labelled frames into a YOLO dataset and explore exactly that dataset, so that our training settings are based on what the data shows.

1. **Dataset preparation:** build `data/cycle-ball-dataset/` with a time-based train/val split, and sanity-check it.
2. **EDA** on `data/cycle-ball-dataset/`.
3. **Conclusions for training:** the settings we will use, each with its reason.

**Run:** open in VS Code or Jupyter with the project's Python and *Run All*. Needs only the repo: the dataset is built from the video and the label export, which are both in git. Figures are saved to `reports/eda/` (not in git, because some of them show video frames).

> **GenAI:** first version generated with Claude Code (Opus 5.5) on 2026-10-29 from our notebook outline (sections 1–3 above). What we checked or changed ourselves: _to fill in_.

## Settings
All paths and numbers used below are set here, so nothing is hidden in the code.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import yaml

# Paths are relative to this notebook (it lives in notebooks/), so ".." is the repo root.
ROOT = Path("..")
# Paths and the train/val split are shared with data/prepare_dataset.py through this config file.
DATASET_CONFIG = yaml.safe_load((ROOT / "configs" / "dataset.yaml").read_text(encoding="utf-8"))
DATASET_DIR = ROOT / DATASET_CONFIG["output_dir"]
# Figures are saved here so we can reuse them in the report.
REPORTS_DIR = ROOT / "reports" / "eda"

# Our video: 3024 x 1078 px and 1170 frames (counted with ffprobe -count_frames).
IMAGE_W, IMAGE_H = 3024, 1078
TOTAL_FRAMES = 1170
# From this frame on, the ball lies still on the floor until the end of the clip (a restart, seen in the video).
RESTART_FRAME = 960
# Labelled frames this close together (in frames) are near-identical: at ~46.8 fps, 5 frames is about 0.1 s.
NEAR_DUPLICATE_GAP = 5

# What we expect in every frame of a 2 vs 2 match.
PLAYERS_ON_COURT = 4
GOALS_ON_COURT = 2
BALLS_IN_PLAY = 1

# Image sizes YOLO could resize to (longest side, in px). 640 is the YOLO default, 1280 is our plan.
IMGSZ_OPTIONS = [640, 1280, 1920]
# Ultralytics' default "scale" augmentation: every training image is randomly zoomed by a factor 1 +/- 0.5.
DEFAULT_AUG_SCALE = 0.5
# The smaller scale we propose in section 3, so the ball doesn't shrink as much.
PROPOSED_AUG_SCALE = 0.25
# COCO calls an object "small" when it is under 32 x 32 px. Small objects are the hardest to detect.
SMALL_OBJECT_PX = 32

# Visual label check: how many random frames to draw per split, with a fixed seed so every run shows the same ones.
CHECK_FRAMES = {"train": 4, "val": 2}
RANDOM_SEED = 0
# Box colours are RGB, because matplotlib shows images as RGB.
CLASS_COLORS = {"player": (0, 255, 0), "ball": (255, 0, 0), "bike": (0, 160, 255), "goal": (255, 0, 255)}
LINE_THICKNESS = 3
LABEL_FONT_SCALE = 1.2
FIGURE_DPI = 150

REPORTS_DIR.mkdir(parents=True, exist_ok=True)


def save_figure(fig, name):
    """Save a figure as reports/eda/<name>.png so we can reuse it in the report."""
    fig.savefig(REPORTS_DIR / f"{name}.png", dpi=FIGURE_DPI, bbox_inches="tight")

## 1. Dataset preparation

**Why we prepare the data ourselves** instead of using the Platform converter's output (`data/labelled/labelled/`):
- **The converter renames every image to a hash,** e.g. `0b094932e8cdfb47c0b310dec545aac0_1.jpg` instead of `frame_00667.jpg`. Then we can no longer tell which part of the video a frame comes from.
- **The split should be our own rule, not a Platform setting.** Neighbouring frames are almost identical, so a frame in train and its neighbour in val leaks information and makes val mAP look better than it is. In the current export, the Platform's val set happens to be frames 13–226, so it is already split by time. But that split is stored per image on the Platform, and we don't control which set frames we label later end up in.
- **Our rule:** frames 1–234 (the first 20% of the video) are val, frames 235–1170 are train. It is set in `configs/dataset.yaml`.
- **Why the start and not the end of the video:** from frame ~960 the referee puts the ball down and it lies still until the end of the clip (a restart). A val set from the last 20% would show that same still ball in every frame and only test that easy case. Section 2.1 shows this.

All the logic is in `data/prepare_dataset.py`, so the notebook and the script can never disagree. It reads the label export, cuts the 73 labelled frames straight out of the video (frame N = the Nth frame of the video), and writes `data/cycle-ball-dataset/` with `images/{train,val}`, `labels/{train,val}` and `data.yaml`. Expected result: **48 train / 25 val** frames.

In [ ]:
# Build data/cycle-ball-dataset/ (see the docstring at the top of the script for how it works).
%run ../data/prepare_dataset.py

### 1b. Sanity check of the prepared dataset

In [ ]:
image_paths = sorted(DATASET_DIR.glob("images/*/*.jpg"))
label_paths = sorted(DATASET_DIR.glob("labels/*/*.txt"))
class_names = yaml.safe_load((DATASET_DIR / "data.yaml").read_text(encoding="utf-8"))["names"]

# Read every line of every label file, split into its values: ["1", "0.51609", "0.56786", ...]
label_lines = []
for path in label_paths:
    for line in path.read_text(encoding="utf-8").splitlines():
        label_lines.append(line.split())

# 1. Every image has a label file with the same name, in the same split.
label_names = {(path.parent.name, path.stem) for path in label_paths}
images_without_label = [path for path in image_paths if (path.parent.name, path.stem) not in label_names]
# 2. Every label line has 5 values: class x_center y_center width height.
bad_lines = [line for line in label_lines if len(line) != 5]
# 3. All coordinates are fractions of the image size, so between 0 and 1.
coordinates = [float(value) for line in label_lines for value in line[1:]]
# 4. All class ids are classes from data.yaml (0-3).
class_ids = {int(line[0]) for line in label_lines}
# 5. Every image has the size of our video.
image_sizes = {cv2.imread(str(path)).shape[:2] for path in image_paths}

print("images:", len(image_paths), "  label files:", len(label_paths), "  boxes:", len(label_lines))
print("1. images without a label file:", len(images_without_label))
print("2. label lines without exactly 5 values:", len(bad_lines))
print("3. coordinates outside 0-1:", sum(not 0 <= value <= 1 for value in coordinates))
print("4. class ids found:", sorted(class_ids), "  classes in data.yaml:", class_names)
print("5. image sizes (height, width):", image_sizes)

**Findings:** 73 images, 73 label files and 785 boxes. Every image has a label file, every label line has 5 values, no coordinate is outside 0–1, the class ids are exactly 0–3, and every image is 3024 × 1078.

**Conclusions:** The prepared dataset is in the format YOLO expects, so we can train on it. These checks only test the format, not whether each box is on the right object; section 2 looks at that.

## 2. EDA on `data/cycle-ball-dataset/`
We read the prepared dataset into two tables, so we analyse exactly the data we will train on:
- `images`: one row per frame (file, split, frame number).
- `boxes`: one row per labelled box (class, position, size).

In [ ]:
# One row per image. The split comes from the folder name, the frame number from the file name.
images = pd.DataFrame({"path": image_paths})
images["file"] = images["path"].apply(lambda path: path.name)
images["split"] = images["path"].apply(lambda path: path.parent.name)
images["frame"] = images["file"].str.extract(r"(\d+)", expand=False).astype(int)
images = images.sort_values("frame").reset_index(drop=True)

# One row per box: read every YOLO label file into a small table, then stack them all.
tables = []
for path in label_paths:
    if path.stat().st_size == 0:
        continue  # an empty label file = a frame without any objects; read_csv can't read it
    table = pd.read_csv(path, sep=" ", header=None, names=["class_id", "x_center", "y_center", "box_w", "box_h"])
    table["file"] = path.stem + ".jpg"
    tables.append(table)
boxes = pd.concat(tables, ignore_index=True)
boxes = boxes.merge(images[["file", "split", "frame"]], on="file")
boxes["class"] = boxes["class_id"].map(class_names)
# Box size in pixels of the original frame.
boxes["box_w_px"] = boxes["box_w"] * IMAGE_W
boxes["box_h_px"] = boxes["box_h"] * IMAGE_H

# The ball gets its own table, because most questions below are about the ball.
balls = boxes[boxes["class"] == "ball"].copy()
boxes.head()

### 2.1 Frames per split, and which part of the video they come from

In [ ]:
images.groupby("split")["frame"].agg(["count", "min", "max"])

In [ ]:
fig, ax = plt.subplots(figsize=(12, 2.5))
for split_name, group in images.groupby("split"):
    ax.scatter(group["frame"], [split_name] * len(group))
ax.set_xlim(0, TOTAL_FRAMES)
ax.set_xlabel("Frame number in the video")
ax.set_ylabel("Split")
ax.set_title("Which frames of the video are labelled, per split")
save_figure(fig, "frames_per_split")
plt.show()

In [ ]:
# How far apart are neighbouring labelled frames? Small gaps mean near-identical frames.
frame_gaps = images["frame"].diff().dropna()
print(frame_gaps.describe().round(1))
print(f"labelled frames at most {NEAR_DUPLICATE_GAP} frames after the previous one:", (frame_gaps <= NEAR_DUPLICATE_GAP).sum())

In [ ]:
# Why val is not the end of the video: the ball in every labelled frame after the restart.
balls[balls["frame"] >= RESTART_FRAME][["frame", "split", "x_center", "y_center"]]

**Findings:**
- Val has 25 frames from frames 13–226, and train has 48 frames from frames 301–1167. The closest frames on either side of the split, 226 (val) and 301 (train), are 75 frames (~1.6 s) apart.
- Neighbouring labelled frames are a median of 10 frames (~0.2 s) apart. 20 of the 73 frames come at most 5 frames after the previous labelled frame.
- From frame 960 on, there are 13 labelled frames, and the ball is in the same place in every one: x = 0.2605–0.2622 and y = 0.7236–0.7280.

**Conclusions:**
- No frame has a near-identical neighbour in the other split, so val isn't inflated by leaked frames.
- Many labelled frames are only a fraction of a second apart, so the 73 frames hold fewer different situations than the number suggests.
- The last table is why val is not the end of the video: a val set from there would be the same still ball 13 times. Those 13 frames are now in train, where they add little variety.

### 2.2 Instances per class, overall and per split

In [ ]:
counts = boxes.groupby(["class", "split"]).size().unstack()
counts["total"] = counts.sum(axis=1)
counts

In [ ]:
fig, ax = plt.subplots()
counts[["train", "val"]].plot(kind="bar", ax=ax, rot=0)
ax.set_xlabel("Class")
ax.set_ylabel("Number of boxes")
ax.set_title("Labelled boxes per class and split")
save_figure(fig, "instances_per_class")
plt.show()

In [ ]:
# Count each class per image (0 if an image has none of it), then the most of each class in one image.
per_image = boxes.groupby(["file", "class"]).size().unstack(fill_value=0)
per_image.max()

**Findings:** 287 player boxes (190 train / 97 val), 279 bike boxes (190 / 89), 146 goal boxes (96 / 50) and 73 ball boxes (48 / 25). No image has more than 1 ball, 2 goals, 4 players or 4 bikes.

**Conclusions:** There's only one ball in play, so the ball can never have more than one box per frame. With 73 frames it can't go above 73 boxes, a quarter of the players. That makes it the smallest class by design, with only 48 training examples. The only way to get more ball examples is to label more frames.

### 2.3 Instances per image
In every frame we expect 4 players, 4 bikes, 2 goals and 1 ball (see Settings).

In [ ]:
fig, ax = plt.subplots()
per_image.sum(axis=1).value_counts().sort_index().plot(kind="bar", ax=ax, rot=0)
ax.set_xlabel("Boxes in the image")
ax.set_ylabel("Number of images")
ax.set_title("Number of labelled boxes per image")
save_figure(fig, "instances_per_image")
plt.show()

In [ ]:
# For each class (column): how many images have 0, 1, 2, ... boxes of it?
per_image.apply(pd.Series.value_counts).fillna(0).astype(int)

In [ ]:
# Images that don't match what we expect. Not always a mistake (a bike can be hidden behind another rider),
# so this is a list to re-check by eye, not a list to delete.
to_check = per_image[(per_image["player"] > PLAYERS_ON_COURT)
                     | (per_image["bike"] < per_image["player"])
                     | (per_image["goal"] != GOALS_ON_COURT)
                     | (per_image["ball"] != BALLS_IN_PLAY)]
to_check.join(images.set_index("file")["split"])

**Findings:**
- 62 of 73 images have the full 11 boxes (4 players, 4 bikes, 2 goals, 1 ball). 5 have 10, 5 have 9, and 1 has 8.
- Every image has exactly 1 ball and 2 goals.
- Players: 68 images have 4 and 5 have 3. Bikes: 62 have 4, 9 have 3 and 2 have 2.
- 7 images are flagged, all in val (frames 22, 32, 34, 39, 53, 80, 102): they have fewer bikes than players. Frame 749 is no longer flagged; it now has 4 players.

**Conclusions:**
- In the flagged val frames, riders are close together in the middle and one bike is hidden behind another. We checked all 7 by eye: each missing bike is more than half hidden, so by our occlusion rule it gets no box. The labels are correct; the flag is how we found the frames worth checking.
- Bikes overlap a lot in crowded moments, so we expect the bike to be harder to detect than the rider there.
- The 5 images with 3 players aren't flagged (a rider can be completely hidden), but they're worth a quick look.

### 2.4 Box sizes per class (in pixels of the 3024 × 1078 frame)

In [ ]:
boxes.groupby("class")[["box_w_px", "box_h_px"]].agg(["min", "median", "max"]).round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for class_name, group in boxes.groupby("class"):
    ax.scatter(group["box_w_px"], group["box_h_px"], s=15, alpha=0.6, label=class_name)
# Log scales: goals are about 10x wider than the ball, so normal axes would squash the ball into a corner.
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Box width (px, log scale)")
ax.set_ylabel("Box height (px, log scale)")
ax.set_title("Box size per class")
ax.legend()
save_figure(fig, "box_sizes")
plt.show()

**How small is the ball when YOLO sees it?** YOLO shrinks the whole frame so that its longest side (the width, 3024 px) becomes `imgsz`, and the ball shrinks by the same factor. During training, the `scale` augmentation can then zoom an image out even further.

In [ ]:
# A round ball has width = height. A motion-blurred ball is drawn longer, so the short side is the real ball size.
ball_size_px = balls[["box_w_px", "box_h_px"]].min(axis=1)
ball_sizes = pd.DataFrame({f"imgsz={size}": ball_size_px * size / IMAGE_W for size in IMGSZ_OPTIONS})

# The scale augmentation zooms out by at most (1 - scale): the default 0.5 -> half size, 0.25 -> 75%.
pd.DataFrame({
    "median ball (px)": ball_sizes.median(),
    "smallest ball (px)": ball_sizes.min(),
    f"smallest, max zoom-out at scale={DEFAULT_AUG_SCALE}": ball_sizes.min() * (1 - DEFAULT_AUG_SCALE),
    f"smallest, max zoom-out at scale={PROPOSED_AUG_SCALE}": ball_sizes.min() * (1 - PROPOSED_AUG_SCALE),
}).round(1)

In [ ]:
fig, ax = plt.subplots()
ball_sizes.boxplot(ax=ax)
ax.axhline(SMALL_OBJECT_PX, color="red", linestyle="--", label=f"COCO 'small object' limit ({SMALL_OBJECT_PX} px)")
ax.set_xlabel("Image size used by YOLO")
ax.set_ylabel("Ball size the model sees (px)")
ax.set_title("Ball size after YOLO resizes the frame")
ax.legend()
save_figure(fig, "ball_size_per_imgsz")
plt.show()

**Findings:**
- Median box size: ball 35 × 34 px, player 105 × 244 px, bike 169 × 149 px, goal 254 × 352 px. Ball boxes are 22–72 px wide and 20–48 px tall.
- In the scatter plot the balls form their own group, far below all other classes.
- The ball's median short side is 6.8 px at `imgsz=640`, 13.7 px at 1280 and 20.5 px at 1920. The smallest ball is 4.2 / 8.4 / 12.6 px.
- With the default `scale=0.5`, training can zoom an image out to half its size, which leaves the smallest ball at 2.1 / 4.2 / 6.3 px. With `scale=0.25` the smallest ball at 1280 stays 6.3 px.
- At every image size, every ball is under the 32 px "small object" limit.

**Conclusions:**
- The median ball is about 3× narrower and 7× shorter than a player.
- At 640 the median ball is ~7 px, which is probably too small to find reliably, so we train and predict at `imgsz=1280`.
- Even at 1280, the default zoom-out can shrink the smallest balls to ~4 px during training. That's smaller than one cell of YOLO's finest detection grid (8 px), so the model would be trained on balls it can hardly see. A smaller `scale` keeps them bigger (section 3).
- These are expectations; the training run will show how well the ball is actually found.

### 2.5 Where objects appear in the frame
Box centres drawn on one frame of the video, per class and per split.

In [ ]:
background = cv2.cvtColor(cv2.imread(str(images["path"].iloc[0])), cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(2, 2, figsize=(16, 6), layout="constrained")
# zip pairs each of the 4 plots with one class.
for ax, (class_name, group) in zip(axes.flatten(), boxes.groupby("class")):
    ax.imshow(background)
    for split_name, split_group in group.groupby("split"):
        ax.scatter(split_group["x_center"] * IMAGE_W, split_group["y_center"] * IMAGE_H, s=15, label=split_name)
    ax.set_title(class_name)
    ax.axis("off")
    ax.legend()
fig.suptitle("Box centres per class")
save_figure(fig, "box_centres_per_class")
plt.show()

In [ ]:
# Horizontal position of the ball (0 = left edge, 1 = right edge of the frame), per split.
balls.groupby("split")["x_center"].agg(["count", "min", "max"]).round(3)

**Findings:**
- Players and bikes appear all over the court, from the left goal to the right goal. In train most of them are on the left half; in val most are in the centre and on the right.
- The goals are in the same place in every frame (the camera doesn't move).
- Ball centres: train from 19% to 52% of the frame width (mostly around the left goal), val from 41% to 55% (the centre, mostly near the near sideline). No labelled ball is further right than 55% of the frame width.

**Conclusions:**
- During training the model never sees the ball on the right side of the court. A horizontal flip turns a ball at the left goal into a ball at the right goal, so flipping is an easy way to fill that gap (section 3). Whether the ball even goes there in this clip, we still have to check on the video.
- Val balls are mostly in places where train has few balls, so val is a fairly hard test for the ball.

### 2.6 Frames without a ball

In [ ]:
frames_without_ball = images[~images["file"].isin(balls["file"])]
print("frames without a ball:", len(frames_without_ball), "of", len(images))

**Findings:** 0 of 73 frames are without a ball.

**Conclusions:** The model never sees a frame without a ball, so it may learn that there is always one. When the ball is hidden, it might then "find" one anyway on something small and red. That's a risk to watch for in the output video. If it happens, we can add 3–5 frames where the ball is completely hidden, with the other objects labelled and no ball box.

### 2.7 Image brightness

In [ ]:
# Mean brightness of each frame: convert to greyscale (0 = black, 255 = white) and average all pixels.
images["brightness"] = images["path"].apply(lambda path: cv2.imread(str(path), cv2.IMREAD_GRAYSCALE).mean())
images.groupby("split")["brightness"].describe().round(1)

In [ ]:
# Brightness inside one frame: the average of each pixel row, from the top of the frame to the bottom.
grey = cv2.imread(str(images["path"].iloc[0]), cv2.IMREAD_GRAYSCALE)
row_brightness = grey.mean(axis=1)

fig, (ax_frames, ax_rows) = plt.subplots(1, 2, figsize=(14, 4))
ax_frames.scatter(images["frame"], images["brightness"])
ax_frames.set_ylim(0, 255)  # the full 0-255 range, so small differences don't look big
ax_frames.set_xlabel("Frame number in the video")
ax_frames.set_ylabel("Mean brightness (0-255)")
ax_frames.set_title("Mean brightness per labelled frame")

ax_rows.plot(row_brightness)
ax_rows.set_ylim(0, 255)
ax_rows.set_xlabel("Pixel row (0 = top of the frame)")
ax_rows.set_ylabel("Mean brightness of the row (0-255)")
ax_rows.set_title(f"Brightness from top to bottom ({images['file'].iloc[0]})")
save_figure(fig, "brightness")
plt.show()

print("darkest row:", row_brightness.min().round(1), "  brightest row:", row_brightness.max().round(1))

**Findings:**
- Mean brightness per frame is 40.3–42.7 on a 0–255 scale (train mean 41.8, val 41.4), with a standard deviation of only 0.5 (train) and 0.1 (val).
- Within one frame, rows go from 2.3 (the stands, almost black) to 129.2 (the court floor near the bottom). Apart from a thin bright line at the very top, roughly the top half of the frame (rows up to ~500) is below 25.

**Conclusions:**
- The hall is dark and lit unevenly: a bright floor, almost black stands.
- But brightness barely changes from frame to frame, so in this clip the lighting is constant. Brightness augmentation therefore isn't needed for variation within this video. It would only help on other footage (another hall or camera), which we can't test yet. Section 3 says what we decide.

### 2.8 Visual label check
Random frames from train and val (fixed seed) with their labelled boxes, to audit label quality by eye. Each figure is also saved to `reports/eda/`.

In [ ]:
def draw_labels(file_name):
    """Return the frame as an RGB image with its labelled boxes and class names drawn on it."""
    path = images.loc[images["file"] == file_name, "path"].iloc[0]
    image = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    for _, box in boxes[boxes["file"] == file_name].iterrows():
        # Box corners in pixels: centre minus/plus half the size.
        x1 = int((box["x_center"] - box["box_w"] / 2) * IMAGE_W)
        y1 = int((box["y_center"] - box["box_h"] / 2) * IMAGE_H)
        x2 = int((box["x_center"] + box["box_w"] / 2) * IMAGE_W)
        y2 = int((box["y_center"] + box["box_h"] / 2) * IMAGE_H)
        color = CLASS_COLORS[box["class"]]
        cv2.rectangle(image, (x1, y1), (x2, y2), color, LINE_THICKNESS)
        cv2.putText(image, box["class"], (x1, y1 - 8), cv2.FONT_HERSHEY_SIMPLEX, LABEL_FONT_SCALE, color, LINE_THICKNESS)
    return image

In [ ]:
check_files = []
for split_name, count in CHECK_FRAMES.items():
    split_images = images[images["split"] == split_name]
    check_files += list(split_images.sample(count, random_state=RANDOM_SEED)["file"])

for file_name in check_files:
    split_name = images.loc[images["file"] == file_name, "split"].iloc[0]
    fig, ax = plt.subplots(figsize=(16, 6))
    ax.imshow(draw_labels(file_name))
    ax.set_title(f"{file_name} ({split_name})")
    ax.axis("off")
    save_figure(fig, f"labels_{Path(file_name).stem}")
    plt.show()

**Findings** (with `RANDOM_SEED = 0` and the current export):
- The 6 frames are train frames 505, 693, 716 and 749, and val frames 32 and 47.
- In all six, the boxes sit tightly on the players, bikes, goals and ball. The referee and the people outside the court aren't boxed, as our labelling rules say.
- Frame 749 now has 4 players.
- In val frame 32, one rider in the middle has no bike box: the bike is more than half hidden (one of the 7 frames from 2.3).

**Conclusions:** In these frames the labels look good enough to train on. Six frames are only a spot check; the systematic check in 2.3 is what finds the frames worth checking by eye.

## 3. Conclusions for training
Each decision follows from a finding above (section number in brackets). Where a decision is the same as Ultralytics' default, we say so, so it's clear which settings we actually change.

| Finding | Decision | Ultralytics default |
|---|---|---|
| The ball is tiny: median 13.7 px at `imgsz=1280`, only 6.8 px at 640 (2.4) | `imgsz=1280` for training and prediction | 640 |
| The default zoom-out (`scale=0.5`) shrinks the smallest ball to 4.2 px, below YOLO's 8 px grid (2.4) | `scale=0.25`: zoom out to 75% at most, so the smallest ball stays 6.3 px | 0.5 |
| No ball on the right side of the court in train; the court is left–right symmetric (2.5) | `fliplr=0.5`: half the training images are mirrored, so a ball at the left goal also teaches the right goal | 0.5 (keep) |
| Fixed, level camera: the goals never move (2.5), and the court never tilts or turns upside down | `degrees=0`, `flipud=0`: no rotation, no vertical flip | 0 / 0 (keep) |
| Dark hall, but brightness hardly changes between frames: 40.3–42.7 out of 255 (2.7) | keep `hsv_v=0.4` (brightness ±40%) for the first run; raise it only when we test on other footage | 0.4 (keep) |
| The ball is red, and colour is one of the few things that sets it apart from the rest of the frame | keep `hsv_h` very small so the red ball stays red; keep `hsv_s=0.7` | 0.015 / 0.7 (keep) |
| Ball: at most 1 box per frame, 48 in train, and by far the smallest objects (2.2, 2.4) | expect the ball to have the lowest AP. If it's too low, try in this order: label more frames (ball on the right side, ball hidden), `imgsz=1920`, cutting the frame into tiles for prediction | – |
| Only 73 frames from one 25 s clip, many of them ~0.2 s apart (2.1) | a small model (nano or small), early stopping with `patience` well below `epochs` (e.g. 20 as a starting value), and the time-based split from section 1 | `patience=100` with `epochs=100`, so no early stopping in practice |
| No frame without a ball (2.6) | watch the output video for "ghost" balls; if we see them, add 3–5 frames where the ball is hidden | – |
| Bikes are often hidden behind other riders; a bike more than half hidden gets no box (2.3) | expect bike recall to be lower than rider recall in crowded moments, and check this in the error analysis | – |

**Where this differs from our outline:**
- **Brightness and saturation:** the outline suggested stronger `hsv_v` and `hsv_s`. The data shows the frames are dark but the lighting is constant, so for this clip we keep the defaults, which already vary brightness by ±40% and saturation by ±70%.
- **Hue:** the default `hsv_h=0.015` is already very small, so we don't need to change it to keep the red ball red.

**Limitations:**
- All frames come from one 25 s clip of one match, filmed by one camera.
- Val is close to the training data (same match, ~1.6 s from the nearest train frame), so a good val score doesn't show that the model works on other matches. We can't test that without other footage.
- Val has only 25 frames (25 balls), so its scores will vary a lot between training runs.